<a href="https://colab.research.google.com/github/rakeshkumar8107/Localrepo/blob/main/project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

TRAIN_DIR = "/content/drive/MyDrive/cig_ps/train_images"
TEST_DIR = "/content/drive/MyDrive/cig_ps/test_images"
LABEL_CSV = "/content/drive/MyDrive/cig_ps/train-labels.csv"

In [6]:
df = pd.read_csv(LABEL_CSV)

df = df[df["text"].astype(str).str.len() == 6].copy()

df = df.reset_index(drop=True)

print("Samples:", len(df))
df.head()

Samples: 19998


,Unnamed: 0,image,text
0,0,train-0.png,BU522X
1,1,train-1.png,XQ8NE2
2,2,train-2.png,DTZD3E
3,3,train-3.png,SM424H
4,4,train-4.png,6YVTQR


In [7]:
all_chars = set()

for text in df["text"]:
    for ch in text:
        all_chars.add(ch)

characters = sorted(list(all_chars))

char_to_num = {
    c:i
    for i,c in enumerate(characters)
}

num_to_char = {
    i:c
    for i,c in enumerate(characters)
}

VOCAB_SIZE = len(characters) + 1

print(characters)
print("Characters:", len(characters))
print("VOCAB_SIZE:", VOCAB_SIZE)

['2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Characters: 31
VOCAB_SIZE: 32


In [8]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42
)

print(len(train_df))
print(len(val_df))

17998
2000


In [9]:
print(characters)
print(VOCAB_SIZE)
print(len(train_df))
print(len(val_df))

['2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
32
17998
2000


In [10]:
IMG_HEIGHT = 100
IMG_WIDTH = 200
MAX_LEN = 6
BATCH_SIZE = 32

In [11]:
sample_df = train_df.iloc[:3000].copy()

print(len(sample_df))
def encode_label(text):
    return [char_to_num[ch] for ch in text]

Y_train = []

for _, row in sample_df.iterrows():
    Y_train.append(
        encode_label(row["text"])
    )

Y_train = np.array(Y_train)

print("Max label:", np.max(Y_train))
print("Min label:", np.min(Y_train))

3000
Max label: 30
Min label: 0


In [12]:
print(encode_label("BU522X"))

[9, 25, 3, 0, 0, 28]


In [13]:
class OCRGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        dataframe,
        image_dir,
        batch_size=32,
        shuffle=True
    ):

        self.df = dataframe.reset_index(drop=True)

        self.image_dir = image_dir

        self.batch_size = batch_size

        self.shuffle = shuffle

        self.on_epoch_end()

    def __len__(self):

        return int(
            np.ceil(
                len(self.df)/self.batch_size
            )
        )

    def on_epoch_end(self):

        self.indexes = np.arange(
            len(self.df)
        )

        if self.shuffle:
            np.random.shuffle(
                self.indexes
            )

    def __getitem__(self,index):

        indexes = self.indexes[
            index*self.batch_size:
            (index+1)*self.batch_size
        ]

        batch = self.df.iloc[indexes]

        images = []
        labels = []

        for _,row in batch.iterrows():

            path = os.path.join(
                self.image_dir,
                row["image"]
            )

            img = cv2.imread(
                path,
                cv2.IMREAD_GRAYSCALE
            )

            img = img.astype(
                np.float32
            )/255.0

            img = np.expand_dims(
                img,
                axis=-1
            )

            images.append(img)

            labels.append(
                encode_label(
                    row["text"]
                )
            )

        return (
            np.array(images),
            np.array(labels)
        )

In [14]:
train_gen = OCRGenerator(
    train_df,
    TRAIN_DIR,
    batch_size=BATCH_SIZE
)

val_gen = OCRGenerator(
    val_df,
    TRAIN_DIR,
    batch_size=BATCH_SIZE
)

In [15]:
X_batch, y_batch = train_gen[0]

print(X_batch.shape)
print(y_batch.shape)

(32, 100, 200, 1)
(32, 6)


In [16]:
print(X_batch.shape)
print(y_batch.shape)
print(y_batch[0])

(32, 100, 200, 1)
(32, 6)
[15  5  8 16 10 24]


In [17]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

inputs = Input(
    shape=(100,200,1),
    name="image"
)

x = Conv2D(
    64,
    (3,3),
    activation="relu",
    padding="same"
)(inputs)

x = MaxPooling2D((2,2))(x)

x = Conv2D(
    128,
    (3,3),
    activation="relu",
    padding="same"
)(x)

x = MaxPooling2D((2,2))(x)

x = Conv2D(
    256,
    (3,3),
    activation="relu",
    padding="same"
)(x)

x = BatchNormalization()(x)

x = MaxPooling2D((2,1))(x)

x = Conv2D(
    512,
    (3,3),
    activation="relu",
    padding="same"
)(x)

x = BatchNormalization()(x)

x = MaxPooling2D((2,1))(x)

In [18]:
shape = x.shape

x = Reshape(
    (
        int(shape[2]),
        int(shape[1] * shape[3])
    )
)(x)

print(x.shape)

(None, 50, 3072)


In [19]:
x = Bidirectional(
    LSTM(
        256,
        return_sequences=True
    )
)(x)

x = Bidirectional(
    LSTM(
        256,
        return_sequences=True
    )
)(x)

In [20]:
outputs = Dense(
    VOCAB_SIZE,
    activation="softmax"
)(x)

prediction_model = Model(
    inputs,
    outputs
)

prediction_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 100, 200, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 100, 200, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 50, 100, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 50, 100, 128)   │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 25, 50, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 25, 50, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 25, 50, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 50, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 50, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 12, 50, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 50, 512)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 50, 3072)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 50, 512)        │     6,817,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 50, 512)        │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 50, 32)         │        16,416 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,962,016 (38.00 MB)

 Trainable params: 9,960,480 (38.00 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [21]:
labels = Input(
    name="labels",
    shape=(MAX_LEN,),
    dtype="float32"
)

input_length = Input(
    name="input_length",
    shape=(1,),
    dtype="int64"
)

label_length = Input(
    name="label_length",
    shape=(1,),
    dtype="int64"
)

In [22]:
from tensorflow.keras import backend as K

def ctc_loss(args):

    y_pred, labels, input_length, label_length = args

    return K.ctc_batch_cost(
        labels,
        y_pred,
        input_length,
        label_length
    )

In [23]:
loss_out = Lambda(
    ctc_loss,
    output_shape=(1,),
    name="ctc"
)(
    [
        prediction_model.output,
        labels,
        input_length,
        label_length
    ]
)

training_model = Model(
    inputs=[
        prediction_model.input,
        labels,
        input_length,
        label_length
    ],
    outputs=loss_out
)

In [41]:
training_model.compile(
    optimizer="adam",
    loss={
        "ctc": lambda y_true,
        y_pred: y_pred
    }
)

In [24]:
training_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 100, 200,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 100, 200,  │        640 │ image[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 50, 100,   │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 50, 100,   │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 25, 50,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 25, 50,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 25, 50,    │      1,024 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 12, 50,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 12, 50,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 50,    │      2,048 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 6, 50,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 50, 3072)  │          0 │ max_pooling2d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 50, 512)   │  6,817,792 │ reshape[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 50, 512)   │  1,574,912 │ bidirectional[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 32)    │     16,416 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ labels (InputLayer) │ (None, 6)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_length        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ label_length        │ (None, 1)         │          0 │ -               

 Total params: 9,962,016 (38.00 MB)

 Trainable params: 9,960,480 (38.00 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [25]:
sample_df = train_df.iloc[:3000]

X_train = []
Y_train = []

for _, row in sample_df.iterrows():

    path = os.path.join(
        TRAIN_DIR,
        row["image"]
    )

    img = cv2.imread(
        path,
        cv2.IMREAD_GRAYSCALE
    )

    img = img.astype(np.float32) / 255.0

    img = np.expand_dims(
        img,
        axis=-1
    )

    X_train.append(img)

    Y_train.append(
        encode_label(
            row["text"]
        )
    )

X_train = np.array(X_train)
Y_train = np.array(Y_train)

print(X_train.shape)
print(Y_train.shape)

(3000, 100, 200, 1)
(3000, 6)


In [30]:
input_lengths = np.ones(
    (len(X_train), 1)
) * 50

label_lengths = np.ones(
    (len(X_train), 1)
) * 6

print(input_lengths.shape)
print(label_lengths.shape)

(3000, 1)
(3000, 1)


In [51]:
dummy = np.zeros(
    (len(X_train),)
)
training_model.compile(
    optimizer="adam",
    loss={
        "ctc": lambda y_true, y_pred: y_pred
    }
)

In [35]:
history = training_model.fit(
    [
        X_train,
        Y_train,
        input_lengths,
        label_lengths
    ],
    dummy,
    batch_size=16,
    epochs=1
)

188/188 ━━━━━━━━━━━━━━━━━━━━ 1199s 6s/step - loss: 23.2783


In [36]:
Y_train = []

for _, row in sample_df.iterrows():
    Y_train.append(
        [char_to_num[ch] for ch in row["text"]]
    )

Y_train = np.array(Y_train)

print(np.max(Y_train))
print(np.min(Y_train))

30
0


In [37]:
input_lengths = np.ones((len(X_train), 1)) * 50
label_lengths = np.ones((len(X_train), 1)) * 6

dummy = np.zeros((len(X_train),))

In [38]:
print(history.history)

{'loss': [23.278268814086914]}


In [39]:
history = training_model.fit(
    [
        X_train,
        Y_train,
        input_lengths,
        label_lengths
    ],
    dummy,
    batch_size=16,
    epochs=10
)

Epoch 1/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1172s 6s/step - loss: 21.8711
Epoch 2/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1203s 6s/step - loss: 21.8123
Epoch 3/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1299s 7s/step - loss: 21.7895
Epoch 4/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1280s 7s/step - loss: 21.7669
Epoch 5/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1266s 7s/step - loss: 21.7313
Epoch 6/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1281s 7s/step - loss: 21.6497
Epoch 7/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1271s 7s/step - loss: 21.3392
Epoch 8/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1286s 7s/step - loss: 21.0546
Epoch 9/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1305s 7s/step - loss: 20.4465
Epoch 10/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 1333s 7s/step - loss: 18.6518


In [54]:
prediction_model.save_weights(
    "crnn_weights.weights.h5"
)

In [55]:
print(history.history["loss"])

[21.871112823486328, 21.812326431274414, 21.789527893066406, 21.76691246032715, 21.731279373168945, 21.649669647216797, 21.339237213134766, 21.05455780029297, 20.446475982666016, 18.651830673217773]


In [56]:
X_val = []
val_texts = []

for _, row in val_df.iloc[:20].iterrows():

    path = os.path.join(
        TRAIN_DIR,
        row["image"]
    )

    img = cv2.imread(
        path,
        cv2.IMREAD_GRAYSCALE
    )

    img = img.astype(np.float32)/255.0

    img = np.expand_dims(
        img,
        axis=-1
    )

    X_val.append(img)

    val_texts.append(
        row["text"]
    )

X_val = np.array(X_val)

print(X_val.shape)

(20, 100, 200, 1)


In [57]:
pred = prediction_model.predict(X_val)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


In [58]:
decoded, _ = tf.keras.backend.ctc_decode(
    pred,
    input_length=np.ones(pred.shape[0]) * pred.shape[1],
    greedy=True
)

decoded = decoded[0].numpy()

In [59]:
for i in range(10):

    text = ""

    for idx in decoded[i]:

        if idx == -1:
            continue

        if idx in num_to_char:
            text += num_to_char[idx]

    print("Actual   :", val_texts[i])
    print("Predicted:", text)
    print()

Actual   : TDJ3F4
Predicted: HA75

Actual   : SEKSCT
Predicted: QTBY867

Actual   : 9EQ4U4
Predicted: QBQAQ

Actual   : U3RBDY
Predicted: TPQ

Actual   : MNYSDG
Predicted: WY8DC

Actual   : 3MZFG6
Predicted: FMAZG

Actual   : VWNBW6
Predicted: WMBM

Actual   : 8UHYWS
Predicted: 2MHMH

Actual   : BHVNX7
Predicted: BHWXM

Actual   : 3WX7WG
Predicted: HWMWM



In [60]:
X_test = []
test_files = []

for file in sorted(os.listdir(TEST_DIR)):

    path = os.path.join(TEST_DIR, file)

    img = cv2.imread(
        path,
        cv2.IMREAD_GRAYSCALE
    )

    img = img.astype(np.float32) / 255.0

    img = np.expand_dims(
        img,
        axis=-1
    )

    X_test.append(img)

    test_files.append(file)

X_test = np.array(X_test)

print(X_test.shape)

(5000, 100, 200, 1)


In [61]:
pred = prediction_model.predict(
    X_test,
    batch_size=32
)

157/157 ━━━━━━━━━━━━━━━━━━━━ 478s 3s/step


In [62]:
decoded, _ = tf.keras.backend.ctc_decode(
    pred,
    input_length=np.ones(pred.shape[0]) * pred.shape[1],
    greedy=True
)

decoded = decoded[0].numpy()

In [63]:
pred_texts = []

for row in decoded:

    text = ""

    for idx in row:

        if idx == -1:
            continue

        if idx in num_to_char:
            text += num_to_char[idx]

    pred_texts.append(text)

In [64]:
submission = pd.DataFrame({
    "image": test_files,
    "prediction": pred_texts
})

submission.to_csv(
    "submission_Rakesh.csv",
    index=False
)

print("Submission created!")

Submission created!


In [65]:
from google.colab import files

files.download(
    "submission_Rakesh.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>